## 13.07 词的相似性和类比任务


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt
import numpy as np

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()
import os
from src.utils import DATA_HUB, DATA_URL, download_extract
from src.pypto_ops import PyPTOMatmul, PyPTOCosineSim

# 预注册词向量数据源（主 notebook 13.7 节已注册，此处幂等）
DATA_HUB['glove.6b.50d'] = (DATA_URL + 'glove.6B.50d.zip',
                           '0b8703943ccdb6eb788e6f091b8946e82231bc4d')
DATA_HUB['wiki.en'] = (DATA_URL + 'wiki.en.zip',
                       'c1816da3821ae9f43899be655002f6c723e91b88')

class TokenEmbedding:
    def __init__(self, embedding_name):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(
            embedding_name)
        self.unknown_idx = 0
        self.token_to_idx = {token: idx for idx, token in
                             enumerate(self.idx_to_token)}
        # 行范数平方一次性预计算（常驻 NPU），knn 每次查询直接复用
        self.w_norm_sq = torch.sum(self.idx_to_vec * self.idx_to_vec,
                                   dim=1, keepdim=True)

    def _load_embedding(self, embedding_name):
        idx_to_token, idx_to_vec = ['<unk>'], []
        data_dir = download_extract(embedding_name)
        # 快速解析：每行只 split 一次，数值部分交给 numpy 的 C 解析
        # （wiki.en 约 251 万词，逐行 float() 解析需 1 小时以上，此处仅需约 2 分钟）
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            for line in f:
                token, rest = line.rstrip().split(' ', 1)
                v = np.fromstring(rest, sep=' ', dtype=np.float32)
                if v.size > 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(v)
        idx_to_vec = np.vstack([np.zeros((1, idx_to_vec[0].size),
                                         dtype=np.float32)] + idx_to_vec)
        # 词向量常驻 NPU（加载时搬一次，knn 查询零搬运）
        device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
        return idx_to_token, torch.tensor(idx_to_vec, device=device)

    def __getitem__(self, tokens):
        indices = [self.token_to_idx.get(token, self.unknown_idx)
                   for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs

    def __len__(self):
        return len(self.idx_to_token)


### 练习 13.7.1

**题目：** 使用 `TokenEmbedding('wiki.en')` 测试 fastText 结果。

**解答：** `wiki.en` 是 fastText 在英文维基百科上预训练的 300 维词向量（含子词信息，能较好处理 OOV 与形态变化）。把主 notebook 中加载 GloVe 的位置换成 `TokenEmbedding('wiki.en')` 即可复用 `knn` / `get_similar_tokens` / `get_analogy`。注意该文件较大（约 2.4GB），下载耗时较长；若网络受限下载失败，下面代码会自动回退到 `glove.6b.50d` 演示相同流程。

以下使用 `torch` 编程进行验证：


In [2]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)

try:
    embed = TokenEmbedding('wiki.en')
    print('已加载 fastText wiki.en')
except Exception as e:
    print(f'wiki.en 下载失败（{e.__class__.__name__}），回退到 GloVe glove.6b.50d 演示：')
    embed = TokenEmbedding('glove.6b.50d')
print(f'词表大小: {len(embed)}')
print(f'向量维度: {embed.idx_to_vec.shape[1]}')

def knn(W, W_norm_sq, x, k):
    # 余弦相似度 = W·x / (‖W‖·‖x‖)
    cos = ((W @ x.reshape(-1, 1)) /
           (W_norm_sq.sqrt() * torch.norm(x) + 1e-9)).reshape(-1)
    topk = torch.topk(cos, k=k)[1]
    return topk.cpu(), cos[topk].cpu()

def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq,
                    embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):
        print(f'{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}')

for token in ['computer', 'good', 'beautiful']:
    print(f'--- 与 "{token}" 最相似的词（fastText wiki.en）---')
    get_similar_tokens(token, 3, embed)

def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq, x, 3)
    for i, c in zip(topk, cos):
        token = embed.idx_to_token[int(i)]
        if token not in (token_a, token_b, token_c) and token != '<unk>':
            return token
    return '<unk>'

for trio in [('man', 'woman', 'son'), ('beijing', 'china', 'tokyo'),
             ('do', 'did', 'go')]:
    print(f'{trio[0]} : {trio[1]} = {trio[2]} : '
          f'{get_analogy(*trio, embed)}')

已加载 fastText wiki.en
词表大小: 2519371
向量维度: 300
--- 与 "computer" 最相似的词（fastText wiki.en）---
computers：cosine相似度=0.839
acomputer：cosine相似度=0.833
•computer：cosine相似度=0.827
--- 与 "good" 最相似的词（fastText wiki.en）---
excellent：cosine相似度=0.722
decent：cosine相似度=0.720
bad：cosine相似度=0.670
--- 与 "beautiful" 最相似的词（fastText wiki.en）---
‘beautiful：cosine相似度=0.827
beautiful,：cosine相似度=0.799
beautifull：cosine相似度=0.798
man : woman = son : daughter
beijing : china = tokyo : japan
do : did = go : went


使用 `PyPTO` 编程进行验证（knn 的余弦相似度由 PyPTOCosineSim 单个 kernel 在 NPU 上完成）：


In [3]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)

def knn(W, W_norm_sq, x, k):
    # 余弦相似度 = W·x / (‖W‖·‖x‖)，由 PyPTOCosineSim 单个 kernel 完成
    # （W 与 W_norm_sq 常驻 NPU，查询零搬运、零重复计算）
    # 词表很大（wiki.en 约 251 万词）时分块调用：超大 M 的 vec kernel
    # 单文件编译耗时过长（数小时级），分块后每块 M=10000 编译秒级
    x = x.reshape(-1, 1)
    block = 10000
    cos = torch.cat([PyPTOCosineSim.apply(W[s:s + block].contiguous(), x,
                                          W_norm_sq[s:s + block]).reshape(-1)
                     for s in range(0, W.shape[0], block)])
    # pypto 暂无 topk 算子包装，topk 在 torch 侧完成；结果一次性取回 CPU
    topk_npu = torch.topk(cos, k=k)[1]
    return topk_npu.cpu(), cos[topk_npu].cpu()

def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq,
                    embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):
        print(f'{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}')

for token in ['computer', 'good', 'beautiful']:
    print(f'--- 与 "{token}" 最相似的词（fastText wiki.en）---')
    get_similar_tokens(token, 3, embed)

def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq, x, 3)
    for i, c in zip(topk, cos):
        token = embed.idx_to_token[int(i)]
        if token not in (token_a, token_b, token_c) and token != '<unk>':
            return token
    return '<unk>'

for trio in [('man', 'woman', 'son'), ('beijing', 'china', 'tokyo'),
             ('do', 'did', 'go')]:
    print(f'{trio[0]} : {trio[1]} = {trio[2]} : '
          f'{get_analogy(*trio, embed)}')

--- 与 "computer" 最相似的词（fastText wiki.en）---


computers：cosine相似度=0.839
acomputer：cosine相似度=0.833
•computer：cosine相似度=0.827
--- 与 "good" 最相似的词（fastText wiki.en）---


excellent：cosine相似度=0.722
decent：cosine相似度=0.720
bad：cosine相似度=0.670
--- 与 "beautiful" 最相似的词（fastText wiki.en）---


‘beautiful：cosine相似度=0.827
beautiful,：cosine相似度=0.799
beautifull：cosine相似度=0.798


man : woman = son : daughter


beijing : china = tokyo : japan


do : did = go : went


### 练习 13.7.2

**题目：** 当词表非常大时，我们怎样才能更快地找到相似的词或完成一个词的类比呢？

**解答：** 暴力法对词表 $\mathcal{V}$ 逐个算余弦相似度是 $O(|\mathcal{V}| \cdot d)$，词表很大（如千万级）时不可接受。加速手段包括：

- **近似最近邻（ANN）**：用 HNSW、乘积量化（PQ）等索引结构，把查询降到 $O(\log |\mathcal{V}|)$ 量级，牺牲少量精度换取数量级的速度提升；
- **局部敏感哈希（LSH）**：把相似的向量以高概率映射到同一桶，桶内再做精确搜索；
- **子词/前缀索引**：利用词形态（如 Trie 树）快速缩小候选集；
- **降维**：对超大词表先用 PCA/随机投影降维再检索，进一步减少每次点积的开销。

**工程实践**：业界常用向量检索库（如 FAISS、Milvus）封装上述 ANN 算法，把“找相似词/类比”变成标准 KNN 查询，避免自己实现。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
